# Métricas de Avaliação para Clustering

Avaliar um agrupamento é mais difícil do que avaliar um classificador: sem rótulos, não há resposta contra a qual comparar a partição encontrada. Resta inferir a qualidade a partir da própria geometria dos dados, medindo o quanto os pontos de um mesmo grupo são próximos entre si (coesão) e o quanto os grupos estão afastados uns dos outros (separação). É isso que as **métricas internas** medem. Neste notebook implementamos três delas e, mais importante, investigamos em que situações elas mentem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.metrics import (silhouette_score as sk_silhouette_score,
                             davies_bouldin_score as sk_davies_bouldin_score,
                             adjusted_rand_score)
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Datasets Sintéticos

Usaremos três conjuntos com topologias deliberadamente diferentes, todos padronizados. O **blobs** traz clusters esféricos e bem separados, o cenário ideal para métodos baseados em centróides. O **circles** traz dois anéis concêntricos, clusters não convexos que compartilham o mesmo centro. E o **moons** traz duas luas entrelaçadas, alongadas e também não convexas.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

datasets = {
    'Blobs':   (StandardScaler().fit_transform(X_blobs), y_blobs),
    'Circles': (StandardScaler().fit_transform(X_circles), y_circles),
    'Moons':   (StandardScaler().fit_transform(X_moons), y_moons),
}

In [ ]:
def plot_clusters(X, labels, ax=None, title=None):
    """Desenha os clusters, com o ruído do DBSCAN (-1) em preto."""
    if ax is None:
        ax = plt.gca()

    unique_labels = np.unique(labels)
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

    for color, label in zip(colors, unique_labels):
        mask = labels == label
        ax.scatter(X[mask, 0], X[mask, 1],
                   color='black' if label == -1 else color,
                   s=40, alpha=0.8, edgecolor='k', linewidth=0.3)

    if title:
        ax.set_title(title)

    return ax


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (X, y)) in zip(axes, datasets.items()):
    plot_clusters(X, y, ax=ax, title=name)

plt.suptitle('Estrutura Real dos Datasets', fontsize=16)
plt.tight_layout()
plt.show()

## Aplicação dos Algoritmos

Rodamos três algoritmos em cada dataset. O K-Means e o aglomerativo recebem o número real de clusters; o DBSCAN recebe `eps` e `min_samples` e descobre o número sozinho.

In [ ]:
clustering_results = {}

for name, (X, y) in datasets.items():
    n_clusters = len(np.unique(y))

    clustering_results[name] = {
        'K-Means': KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit_predict(X),
        'Agglomerative': AgglomerativeClustering(n_clusters=n_clusters).fit_predict(X),
        'DBSCAN': DBSCAN(eps=0.4, min_samples=4).fit_predict(X),
    }

In [ ]:
fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 15))

for i, (dataset_name, results) in enumerate(clustering_results.items()):
    X, _ = datasets[dataset_name]

    for j, (algo_name, labels) in enumerate(results.items()):
        plot_clusters(X, labels, ax=axes[i, j],
                      title=algo_name if i == 0 else None)
        if j == 0:
            axes[i, j].set_ylabel(dataset_name, fontsize=14, fontweight='bold')

plt.suptitle('Resultados dos Algoritmos', fontsize=16)
plt.tight_layout()
plt.show()

Visualmente o diagnóstico é imediato: nos blobs os três acertam; nos círculos e nas luas, apenas o DBSCAN recupera a estrutura real, enquanto o K-Means e o aglomerativo cortam os grupos com fronteiras que ignoram a forma dos dados. Guarde essa impressão, porque as métricas vão discordar dela.

## Silhouette Score

O Silhouette compara, para cada ponto, o quanto ele está próximo dos colegas de cluster com o quanto está próximo do cluster vizinho mais próximo. Varia de -1 a +1, e **quanto maior, melhor**.

Para um ponto $i$:

$$ s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} $$

- $a(i)$ é a distância média de $i$ aos demais pontos do seu próprio cluster (coesão):
$$ a(i) = \frac{1}{|C_k| - 1} \sum_{j \in C_k, \, j \neq i} d(i, j), \quad i \in C_k $$
- $b(i)$ é a menor distância média de $i$ a um cluster do qual não faz parte (separação):
$$ b(i) = \min_{m \neq k} \left( \frac{1}{|C_m|} \sum_{j \in C_m} d(i, j) \right) $$

O score do conjunto é a média de $s(i)$ sobre todos os pontos.

In [ ]:
def silhouette_score(X, labels):
    """Silhouette médio, descartando os pontos de ruído (-1)."""
    D = squareform(pdist(X))
    cluster_labels = np.unique(labels)
    cluster_labels = cluster_labels[cluster_labels != -1]

    silhouette_values = []

    for i in range(len(X)):
        current = labels[i]

        if current == -1:
            continue

        # a(i): distância média aos colegas de cluster, excluindo o próprio ponto
        same_cluster = labels == current
        same_cluster[i] = False
        a_i = np.mean(D[i][same_cluster]) if same_cluster.sum() > 0 else 0

        # b(i): menor distância média a um cluster do qual i não faz parte
        b_i = np.inf
        for other in cluster_labels:
            if other != current:
                b_i = min(b_i, np.mean(D[i][labels == other]))

        denominator = max(a_i, b_i)
        silhouette_values.append((b_i - a_i) / denominator if denominator > 0 else 0)

    return np.mean(silhouette_values)

## Davies-Bouldin Index

O Davies-Bouldin resume cada cluster a duas quantidades: a dispersão em torno do seu centróide e a distância até os outros centróides. Para cada cluster, toma o pior par possível; o índice é a média desses piores casos. **Quanto menor, melhor.**

$$ DBI = \frac{1}{K} \sum_{k=1}^{K} \max_{m \neq k} R_{km}, \qquad R_{km} = \frac{S_k + S_m}{D_{km}} $$

- $S_k$ é a dispersão média do cluster $C_k$ em torno do seu centróide $\mu_k$:
$$ S_k = \frac{1}{|C_k|} \sum_{x \in C_k} \|x - \mu_k\|_2 $$
- $D_{km}$ é a distância entre os centróides:
$$ D_{km} = \|\mu_k - \mu_m\|_2 $$

Repare onde $D_{km}$ aparece: no denominador. Dois clusters com centróides próximos levam $R_{km}$ ao infinito, independentemente de quão bem separados estejam de fato.

In [ ]:
def davies_bouldin_score(X, labels):
    """Davies-Bouldin Index, descartando os pontos de ruído (-1)."""
    cluster_labels = np.unique(labels)
    cluster_labels = cluster_labels[cluster_labels != -1]
    n_clusters = len(cluster_labels)

    centroids = np.array([X[labels == c].mean(axis=0) for c in cluster_labels])
    dispersions = np.array([np.linalg.norm(X[labels == c] - centroid, axis=1).mean()
                            for c, centroid in zip(cluster_labels, centroids)])

    db_index = 0

    for i in range(n_clusters):
        worst = 0

        for j in range(n_clusters):
            if i == j:
                continue

            similarity = (dispersions[i] + dispersions[j]) / np.linalg.norm(centroids[i] - centroids[j])
            worst = max(worst, similarity)

        db_index += worst

    return db_index / n_clusters

## Dunn Index

O Dunn não usa centróides nem médias: compara a **menor** distância entre pontos de clusters diferentes com o **maior** diâmetro entre os clusters. **Quanto maior, melhor.**

$$ D = \frac{\min_{1 \le k < m \le K} d(C_k, C_m)}{\max_{1 \le l \le K} \text{diam}(C_l)} $$

- $d(C_k, C_m)$ é a distância entre os dois pontos mais próximos de clusters diferentes:
$$ d(C_k, C_m) = \min_{x \in C_k, \, y \in C_m} \|x - y\|_2 $$
- $\text{diam}(C_l)$ é a distância entre os dois pontos mais distantes de um mesmo cluster:
$$ \text{diam}(C_l) = \max_{x, y \in C_l} \|x - y\|_2 $$

É uma medida de conectividade, não de compacidade, seguindo a mesma lógica da ligação simples. E, como depende de apenas quatro pontos do conjunto inteiro, é bastante sensível a outliers.

In [ ]:
def dunn_index(X, labels):
    """Dunn Index, descartando os pontos de ruído (-1)."""
    D = squareform(pdist(X))
    cluster_labels = np.unique(labels)
    cluster_labels = cluster_labels[cluster_labels != -1]

    indices = [np.where(labels == c)[0] for c in cluster_labels]

    # maior diâmetro entre os clusters
    max_intra = max((D[np.ix_(idx, idx)].max() for idx in indices if len(idx) > 1), default=0)

    if max_intra == 0:
        return np.inf

    # menor distância entre pontos de clusters diferentes
    min_inter = np.inf
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            min_inter = min(min_inter, D[np.ix_(indices[i], indices[j])].min())

    return min_inter / max_intra

### Comparação com o Scikit-Learn

O Scikit-Learn traz o Silhouette e o Davies-Bouldin prontos (o Dunn não faz parte da biblioteca). Vale conferir se nossas implementações batem.

```python
from sklearn.metrics import silhouette_score, davies_bouldin_score

silhouette_score(X, labels)      # quanto maior, melhor
davies_bouldin_score(X, labels)  # quanto menor, melhor
```

In [ ]:
for dataset_name, results in clustering_results.items():
    for algo_name, labels in results.items():
        X, _ = datasets[dataset_name]
        print(f"{dataset_name:8} {algo_name:14}"
              f"  silhouette {silhouette_score(X, labels):6.3f} / {sk_silhouette_score(X, labels):6.3f}"
              f"   davies-bouldin {davies_bouldin_score(X, labels):7.3f} / {sk_davies_bouldin_score(X, labels):7.3f}")

Os valores coincidem, mas por um motivo específico: com `eps=0.4` o DBSCAN não marcou nenhum ponto como ruído nesses datasets. Quando marca, as implementações divergem, já que nós descartamos os pontos com rótulo -1 enquanto o Scikit-Learn os trata como se fossem mais um cluster.

In [ ]:
X_c, _ = datasets['Circles']
noisy_labels = DBSCAN(eps=0.25, min_samples=4).fit_predict(X_c)

print(f"pontos de ruído: {np.sum(noisy_labels == -1)}")
print(f"nosso   (descarta o ruído):     {silhouette_score(X_c, noisy_labels):.3f}")
print(f"sklearn (ruído como cluster):   {sk_silhouette_score(X_c, noisy_labels):.3f}")

Nenhuma das duas convenções é obviamente certa. Descartar o ruído mede a qualidade dos clusters encontrados, mas premia um algoritmo que joga fora tudo que é difícil; tratá-lo como cluster pune o DBSCAN por um rótulo que, por definição, não descreve um grupo. O que não se pode é comparar números calculados sob convenções diferentes.

## Análise Comparativa

Com as três métricas implementadas, montamos a tabela completa.

In [ ]:
evaluation_results = []

for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]

    for algo_name, labels in results.items():
        evaluation_results.append({
            'Dataset': dataset_name,
            'Algoritmo': algo_name,
            'Silhouette': silhouette_score(X, labels),
            'Davies-Bouldin': davies_bouldin_score(X, labels),
            'Dunn': dunn_index(X, labels),
        })

results_df = pd.DataFrame(evaluation_results)

results_df.style.background_gradient(
    cmap='viridis', subset=['Silhouette', 'Dunn']
).background_gradient(
    cmap='viridis_r', subset=['Davies-Bouldin']
).format({'Silhouette': '{:.3f}', 'Davies-Bouldin': '{:.3f}', 'Dunn': '{:.3f}'})

### O Que a Tabela Revela

Nos blobs os três algoritmos chegam à mesma partição, e as três métricas concordam. As outras duas linhas é que interessam.

Nos círculos, o DBSCAN recupera exatamente os dois anéis e o K-Means os corta ao meio com uma reta. Ainda assim, o Silhouette **prefere o K-Means** (0,323 contra 0,148) e o Davies-Bouldin declara o DBSCAN catastrófico (95,4 contra 1,23). Só o Dunn acerta: 0,102 contra 0,014. Nas luas repete-se a inversão, com Silhouette de 0,494 para o K-Means contra 0,380 para o DBSCAN, que é quem acerta a estrutura.

A explicação está no que cada índice mede.

O **Silhouette** usa distâncias médias. Num anel, o ponto diametralmente oposto é colega de cluster e está longuíssimo, inflando o $a(i)$; o ponto do anel vizinho logo ao lado está perto e deprime o $b(i)$. A silhueta desaba mesmo com a partição perfeita.

O **Davies-Bouldin** é ainda mais frágil aqui, porque resume cada cluster ao seu centróide. Os dois anéis são concêntricos: seus centróides ficam a 0,027 de distância um do outro. Como esse valor é o denominador de $R_{km}$, o índice explode. O 95,4 não diz que o agrupamento é ruim, diz que a suposição do índice não vale para esses dados.

O **Dunn** escapa porque mede conectividade, e não compacidade: pergunta se existe um vão entre os grupos, não se cada grupo é redondo. Por isso enxerga anéis e luas. O preço é a fragilidade oposta: como depende de apenas quatro pontos, um outlier no lugar errado derruba o índice inteiro.

A lição não é que o Dunn seja melhor. É que **Silhouette e Davies-Bouldin embutem a hipótese de clusters convexos e isotrópicos**, que é exatamente a hipótese do K-Means. Usá-los para escolher entre algoritmos tende a premiar quem faz essa suposição, valha ela ou não nos dados.

## Métricas Externas

Como esses datasets são sintéticos, temos os rótulos verdadeiros e podemos medir diretamente a distância entre a partição encontrada e a estrutura real. O **Adjusted Rand Index (ARI)** olha todos os pares de pontos e conta em quantos as duas partições concordam (juntos em ambas, ou separados em ambas), corrigindo o resultado pelo que se esperaria do acaso. Vale 1 para partições idênticas, 0 para um agrupamento aleatório, e fica negativo para um pior que o acaso.

Como não tem opinião sobre a forma dos clusters, o ARI serve de árbitro para o desacordo da seção anterior.

In [ ]:
for dataset_name, results in clustering_results.items():
    X, y = datasets[dataset_name]

    scores = "   ".join(f"{name} {adjusted_rand_score(y, labels):6.3f}"
                        for name, labels in results.items())
    print(f"{dataset_name:8} {scores}")

O ARI confirma a leitura visual e contradiz as métricas internas. Nos círculos, o K-Means tira -0,003: literalmente indistinguível de um sorteio, apesar do Silhouette de 0,323 que o colocava em primeiro. Nas luas, o ARI ordena o aglomerativo (0,716) acima do K-Means (0,479), invertendo de novo a ordem do Silhouette. O DBSCAN tira 1,000 nos dois casos.

Só que esse árbitro não existe num problema real: se tivéssemos os rótulos, não estaríamos clusterizando. É precisamente por isso que as métricas internas são necessárias, e precisamente por isso é preciso saber o que elas assumem antes de confiar nelas.

## Limitações

Toda métrica interna é um viés declarado. Ela precisa definir o que conta como um bom cluster, e essa definição nunca é neutra: Silhouette e Davies-Bouldin premiam grupos compactos e convexos, enquanto o Dunn premia grupos conectados e bem separados. Escolher a métrica é escolher a hipótese.

O custo também difere. Silhouette e Dunn precisam da matriz de distâncias completa, $O(N^2)$, o que limita o uso a alguns milhares de pontos. O Davies-Bouldin é $O(NK)$, porque só compara pontos a centróides, e essa é parte da razão de ser tão popular e a mesma razão de ser tão dependente de centróides.

Há ainda três armadilhas de interpretação. O ruído não tem lugar definido: descartar ou não os pontos com rótulo -1 muda o valor, e não existe convenção universal. Comparar valores entre datasets não significa nada, porque as escalas dependem da dimensão, do número de clusters e da densidade dos dados, de modo que só faz sentido comparar partições do mesmo conjunto. E nenhuma delas testa se existe estrutura: aplicadas a pontos uniformemente aleatórios, as três devolvem números, e algum $K$ sempre parecerá melhor que os outros. Elas comparam partições, não dizem se agrupar fazia sentido.

## Exercícios

### Exercício 1: Sensibilidade aos Parâmetros

Varie os parâmetros dos três algoritmos (`n_clusters` no K-Means e no aglomerativo, `eps` e `min_samples` no DBSCAN) e recalcule as três métricas nos mesmos datasets.

Observe em particular dois efeitos: o que acontece com cada métrica quando o número de clusters passa do valor real, e o que acontece com o Dunn quando o DBSCAN começa a marcar pontos como ruído.

In [ ]:
# Seu código aqui

### Exercício 2: Avaliação no Dataset Iris

Aplique os três algoritmos ao dataset Iris, lembrando de padronizar os dados antes, e avalie os resultados com as três métricas internas.

Em seguida, calcule o ARI contra as espécies reais e compare: alguma métrica interna elege o mesmo vencedor que o ARI? Discuta o resultado à luz do que cada índice assume sobre a forma dos clusters e do fato de que duas das três espécies do Iris se sobrepõem.

In [ ]:
# Seu código aqui